# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df.trend_direction == "down").astype(int)
d = df[df.avg_position > 0].copy()  # avg_position == 0 means "no data" — excluded
print(d.shape)


## 1. Method choice and why

**Random Forest Classifier**, scored by `predict_proba` and ranked — matching the lane's task type (ranking/scoring). I'm not choosing Logistic Regression because the earlier signal audit (Week 4) showed decline isn't explained by one or two clean linear relationships: CTR-vs-position was confirmed but staleness was MIXED/non-monotonic, suggesting real interaction effects between features rather than a single linear trend. A Random Forest can pick up on those interactions without me hand-engineering them, while staying far more interpretable (via feature importance) than Gradient Boosting, which isn't needed yet at this data size. I excluded `trend_direction` and `trend_pct` from the feature set since they define the label.

In [ ]:
features = ['content_age_days','days_since_last_update','impressions_90d','avg_position',
            'ctr','word_count','search_volume']
print('Features used:', features)
print('Excluded (label source):', ['trend_direction','trend_pct'])


## 2. Split design

**Grouped by client**, not random row-shuffling: a `GroupShuffleSplit`-style split where every page from a given client goes entirely into train OR test, never both. This is the honest split for this question because the real deployment scenario is scoring pages for clients the model may not have been tuned on — a random row split would let the model see other pages from the same client in training, leaking client-specific patterns (industry, content style, baseline traffic level) into the test score. This is the same split design used for the Week 1-2 baseline comparisons, reused here on purpose so the Week-4 baseline and this model are compared fairly on identical held-out clients.

In [ ]:
from sklearn.model_selection import train_test_split

clients = d['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train = d[d.client_id.isin(train_clients)]
test  = d[d.client_id.isin(test_clients)].copy()
print(f"train clients: {len(train_clients)}  rows: {len(train)}")
print(f"test clients:  {len(test_clients)}  rows: {len(test)}")


## 3. Train + compare vs my baseline

Same test split, same metric (Precision@20 / Precision@50), same label (`is_declining_label`) as the Week-4 `ctr_below_position_benchmark` baseline. The baseline score is recomputed here on the identical train/test split for a fair comparison.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline: recompute the Week-4 ctr_below_position_benchmark rule on this same split ---
benchmark = train.groupby('position_tier')['ctr'].mean()  # benchmark learned from TRAIN only
test['benchmark_ctr'] = test['position_tier'].map(benchmark)
test['ctr_gap'] = (test['benchmark_ctr'] - test['ctr']).clip(lower=0)
test['baseline_score'] = test['ctr_gap'] * test['impressions_90d']

# --- Model: Random Forest on honest features, grouped-holdout ---
from sklearn.ensemble import RandomForestClassifier
Xtr = train[features].replace([np.inf, -np.inf], np.nan).fillna(0)
Xte = test[features].replace([np.inf, -np.inf], np.nan).fillna(0)
ytr = train['is_declining_label'].values
yte = test['is_declining_label'].values

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
test['model_score'] = rf.predict_proba(Xte)[:, 1]

print(f"{'k':>4} {'baseline':>10} {'random_forest':>15}")
for k in (20, 50):
    bp = precision_at_k(test['baseline_score'].values, yte, k)
    mp = precision_at_k(test['model_score'].values, yte, k)
    print(f"{k:>4} {bp:>10.3f} {mp:>15.3f}")


## 4. Errors and interpretation

Feature importance + a look at the model's actual top-20 errors (false positives), not just a metric number.

In [ ]:
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importance:")
print(imp.round(3))
print()

top20 = test.sort_values('model_score', ascending=False).head(20)
print("Top 20 label mix (1=declining, 0=not):", top20['is_declining_label'].value_counts().to_dict())

false_pos = top20[top20['is_declining_label'] == 0]
print()
print("False positives in top 20 (model flagged, not actually declining):")
print(false_pos[['content_id','content_age_days','impressions_90d','avg_position',
                  'trend_direction']].to_string(index=False))
print()
print("Interpretation: content_age_days and impressions_90d dominate importance. All the ",
      "false positives above are young pages (low content_age_days) labeled 'stable', not ",
      "'down' — consistent with the Week-1/2 finding that younger pages skew toward 'down' ",
      "more often overall, so the model over-weights youth as a decline signal and ",
      "over-flags young-but-currently-stable pages. This is directional, not proof the model ",
      "understands causation — it's reproducing a correlation, including its false positives.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.